# RAG System Deep Dive: From Start to Finish

This guide walks you through exactly how the Retrieval-Augmented Generation (RAG) system works. We'll follow a document as it goes through the whole process: being loaded and broken down, stored in a database, searched, and finally used to answer a question.

## What we're trying to achieve
- **Context**: Keeping the meaning of documents by breaking them into overlapping pieces.
- **Speed**: Using a smart indexing system that only updates when your files change.
- **Accuracy**: Using the best vector stores to find exactly what you're looking for.
- **Truth**: Making sure the LLM only answers using the info it finds in your documents.

---
## Step 1: Loading and Indexing Documents

First, we take raw files (PDFs and text) and turn them into a searchable database. This is handled by the `src/ingestion.py` and `src/vectorstores.py` modules.

### 1.1 Extracting and Cutting up Text
We load the files and then cut them into smaller, overlapping segments. We overlap the pieces so that no information gets cut in half at the edges.

In [ ]:
import sys
sys.path.append('.') # Access the root folder

from src.config import AppConfig
from src.ingestion import load_text_and_pdfs, split_documents

# 1. Setup config and choose where the data is
cfg = AppConfig()
print(f"Looking for data in: {cfg.paths.data_dir}")

# 2. Find and load all files
raw_docs = load_text_and_pdfs(cfg.paths.data_dir)
print(f"Found {len(raw_docs)} original documents.")

# 3. Break them into smaller pieces (chunks)
chunks = split_documents(raw_docs, chunk_size=500, chunk_overlap=100)
print(f"Created {len(chunks)} text segments.")

### 1.2 Creating Vectors and Storing Them
Each piece of text is turned into a list of numbers (an embedding) that represents its meaning. These are saved in a local **Chroma** database.

**Deduplication**: We use a hashing trick so the system knows if a file has already been indexed. This makes it much faster after the first run.

In [ ]:
from src.embeddings import EmbeddingManager
from src.vectorstores import ChromaVectorStore

# 1. Setup the embedding model
embed_manager = EmbeddingManager()
embed_manager.load_model()

# 2. Setup the database
vector_store = ChromaVectorStore()
print(f"Database currently has {vector_store.count()} records.")

# 3. Turn text into vectors
texts = [doc.page_content for doc in chunks]
embeddings = embed_manager.generate_embeddings(texts)

# 4. Add the new pieces to the database
vector_store.add_documents(chunks, embeddings)
print(f"Database now has {vector_store.count()} records.")

---
## Step 2: Searching for Information

Now we need to find the most relevant pieces of text based on what the user asks. This system supports **Chroma** (local) and **Typesense** (remote).

### 2.1 Local Search (Chroma)
We use 

In [ ]:
from src.retrieval import ChromaRAGRetriever

retriever = ChromaRAGRetriever(vector_store, embed_manager)

# Input a search term here to find relevant chunks in your database
query = "Search query goes here"
results = retriever.retrieve(query, top_k=2)

print(f"--- SEARCH RESULTS (CHROMA) ---\n")
for i, res in enumerate(results):
    print(f"Result {i+1} (Score: {res['similarity_score']:.4f}):")
    print(f"{res['content'][:250]}...\n")

### 2.2 Remote Search (Typesense)
For bigger projects, you can use Typesense, which handles searching on a remote server.

In [ ]:
from src.vectorstores import TypesenseVectorStore
from src.retrieval import TypesenseRAGRetriever

if not cfg.typesense.api_key:
    print("Typesense API key not found. Skipping this part.")
else:
    ts_store = TypesenseVectorStore()
    ts_retriever = TypesenseRAGRetriever(ts_store)
    
    print("Typesense search engine is ready.")

---
## Step 3: Generating the Answer

Finally, we take the text we found and give it to the LLM so it can write a clear answer.

### 3.1 Putting it all together
The `AdvancedRAGPipeline` handles the whole flow. It builds the prompt, makes sure the LLM sticks to the facts, and adds citations for you.

In [ ]:
from src.pipeline import AdvancedRAGPipeline, build_groq_llm

# 1. Setup the Groq LLM
llm = build_groq_llm()

# 2. Connect everything to the pipeline
pipeline = AdvancedRAGPipeline(retriever=retriever, llm=llm)

# 3. Run a final test query based on your documents
query = "Your final question here"
result = pipeline.query(query, top_k=3, min_score=0.1, summarize=True)

print(f"--- FINAL RESULT ---\n")
print(f"ANSWER:\n{result['answer']}")

if result['summary']:
    print(f"\nSUMMARY:\n{result['summary']}")

## Summary

That's it! You've seen how the system loads data, searches for the right info, and generates an answer. By keeping these steps separate, it's easy to adjust or improve each part of the process.